# Clustering IO & Apply

Save a clustering to JSON, load it later, and apply it to new data.
This is useful when you want to cluster once and reuse the same
clustering across multiple datasets.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook"

da = sample_energy_data(n_days=30).sel(region="north", scenario="low")
print(f"Shape: {dict(da.sizes)}")

## Aggregate and save clustering

In [ ]:
result = tsam_xarray.aggregate(
    da,
    time_dim="time",
    cluster_dim="variable",
    n_clusters=4,
)

# Save the clustering to JSON
result.clustering.to_json("clustering.json")
print(f"Saved clustering: {result.n_clusters} clusters")
result.typical_periods.plotly.line(
    x="timestep",
    color="cluster",
    facet_col="variable",
    title="Original clustering",
)

## Load and apply to new data

Load the saved clustering and apply it to different data.
The clustering assignments are reused — the new data gets
the same cluster structure.

In [ ]:
import numpy as np

# Load the clustering
clustering = tsam_xarray.load_clustering("clustering.json")
print(f"Loaded: time_dim={clustering.time_dim}")
print(f"  cluster_dim={clustering.cluster_dim}")

# Apply to new data (same shape, different values)
da_new = da.copy(data=np.random.default_rng(99).random(da.shape))
new_result = clustering.apply(da_new)
print(f"Applied: {new_result.n_clusters} clusters")
print(f"  is_transferred={new_result.is_transferred}")

In [ ]:
new_result.typical_periods.plotly.line(
    x="timestep",
    color="cluster",
    facet_col="variable",
    title="Applied clustering (new data)",
)

## Compare accuracy

In [ ]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "original": result.accuracy.rmse.to_series(),
        "applied": new_result.accuracy.rmse.to_series(),
    }
)
comparison

## Apply with different cluster_dim

You can apply the clustering to data with renamed dimensions:

In [ ]:
da_renamed = da_new.rename({"variable": "source"})
result_renamed = clustering.apply(da_renamed, cluster_dim="source")
print(f"Dims: {result_renamed.typical_periods.dims}")
result_renamed.accuracy.rmse.to_dataframe("RMSE")